In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style='whitegrid')
df = pd.read_csv('data/merc.csv')
print(df.shape)
print(df.columns.tolist())
print(df.dtypes)
display(df.head(3))

In [ ]:
print(df.describe(include='all').transpose().to_string())
print('--- missing ---')
print(df.isna().sum().to_string())
print('--- cat ---')
for c in ['model','transmission','fuelType']:
    print(c, df[c].nunique())
    print(df[c].value_counts().head(10).to_string())
    print()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df["price"], bins=80)
ax.set_title("price hist")
ax.set_xlabel("price")
plt.tight_layout()
plt.show()
print(df["price"].quantile([0, 0.25, 0.5, 0.75, 0.95, 0.99, 1.0]).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
df["model"].value_counts().head(10).plot(kind="bar", ax=ax)
ax.set_title("top-10 model")
plt.tight_layout()
plt.show()

In [ ]:
df["car_age"] = 2021 - df["year"]
corr = df[["price","year","car_age","mileage","tax","mpg","engineSize"]].corr(numeric_only=True)
print(corr["price"].sort_values(ascending=False).to_string())
fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(df["mileage"], df["price"], s=2, alpha=0.3)
ax.set_title("mileage vs price")
plt.tight_layout()
plt.show()

In [ ]:
import sys
sys.path.insert(0, "src")
from preprocess import load_clean, train_val_test, build_preprocessor, split_xy
df = load_clean("data/merc.csv")
tra, va, te = train_val_test(df)
print(df.shape)
print(tra.shape, va.shape, te.shape)

In [ ]:
Xtr, ytr = split_xy(tra)
Xva, yva = split_xy(va)
Xte, yte = split_xy(te)
pre = build_preprocessor()
Xtr_t = pre.fit_transform(Xtr)
Xva_t = pre.transform(Xva)
Xte_t = pre.transform(Xte)
print(Xtr_t.shape, Xva_t.shape, Xte_t.shape)

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
bmean = DummyRegressor(strategy="mean")
bmean.fit(Xtr_t, ytr)
bmed = DummyRegressor(strategy="median")
bmed.fit(Xtr_t, ytr)
for name, m in [("mean", bmean), ("median", bmed)]:
    pr_tr = m.predict(Xtr_t)
    pr_va = m.predict(Xva_t)
    pr_te = m.predict(Xte_t)
    print(name, round(mean_absolute_error(ytr, pr_tr), 1), round(mean_absolute_error(yva, pr_va), 1), round(mean_absolute_error(yte, pr_te), 1))
    print(name, round(mean_squared_error(ytr, pr_tr), 1), round(r2_score(yte, pr_te), 4))

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV, validation_curve
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
gs = GridSearchCV(Ridge(), {"alpha": [0.1, 1.0, 10.0, 100.0, 1000.0]}, cv=5, scoring="neg_mean_absolute_error")
gs.fit(Xtr_t, ytr)
print(gs.best_params_)
print(round(-gs.best_score_, 1))
best = gs.best_estimator_
ptr = best.predict(Xtr_t)
pva = best.predict(Xva_t)
pte = best.predict(Xte_t)
print(round(mean_absolute_error(ytr, ptr), 1), round(mean_absolute_error(yva, pva), 1), round(mean_absolute_error(yte, pte), 1))
print(round(mean_squared_error(yte, pte), 1), round(r2_score(yte, pte), 4))
tr_s, va_s = validation_curve(Ridge(), Xtr_t, ytr, param_name="alpha", param_range=[0.1, 1.0, 10.0, 100.0, 1000.0], cv=5, scoring="neg_mean_absolute_error")
fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogx([0.1, 1.0, 10.0, 100.0, 1000.0], -va_s.mean(axis=1))
ax.set_xlabel("alpha")
ax.set_ylabel("val MAE")
plt.tight_layout()
plt.show()

In [ ]:
import sys
sys.path.insert(0, "src")
from ridge_gd import MyRidgeGD
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
ytrn = ytr.to_numpy()
yvan = yva.to_numpy()
yten = yte.to_numpy()
results = []
for lr in [0.1, 0.2]:
    for al in [1.0, 5.0]:
        m = MyRidgeGD(lr=lr, n_iter=600, alpha=al, batch_size=256, random_state=42)
        m.fit(Xtr_t, ytrn, Xva_t, yvan)
        vm = mean_absolute_error(yvan, m.predict(Xva_t))
        results.append((vm, lr, al, m))
        print(lr, al, round(vm, 1))
results.sort(key=lambda t: t[0])
bl, blr, bal, own = results[0]
print(blr, bal, round(bl, 1))
po_tr = own.predict(Xtr_t)
po_va = own.predict(Xva_t)
po_te = own.predict(Xte_t)
print(round(mean_absolute_error(ytrn, po_tr), 1), round(mean_absolute_error(yvan, po_va), 1), round(mean_absolute_error(yten, po_te), 1))
print(round(mean_squared_error(yten, po_te), 1), round(r2_score(yten, po_te), 4))
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(own.hist_train_, label="train")
ax.plot(own.hist_val_, label="val")
ax.set_xlabel("epoch")
ax.set_ylabel("MAE")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
pb = bmed.predict(Xte_t)
ps = best.predict(Xte_t)
po = own.predict(Xte_t)
rows = [("base", pb), ("sklearn", ps), ("own", po)]
for name, p in rows:
    print(name, round(mean_absolute_error(yte, p), 1), round(mean_squared_error(yte, p), 1), round(r2_score(yte, p), 4))
v = [mean_absolute_error(yte, p) for _, p in rows]
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar([r[0] for r in rows], v)
ax.set_ylabel("test MAE")
plt.tight_layout()
plt.show()